In [0]:
CREATE OR REPLACE VIEW workspace.default.inspections_clean_v2 AS
WITH filtered_and_deduplicated AS (
  SELECT DISTINCT
    CAST(camis AS STRING)                      AS restaurant_id,
    dba                                        AS restaurant_name,
    INITCAP(TRIM(boro))                        AS borough,
    LPAD(CAST(zipcode AS STRING), 5, '0')      AS zip_code,
    cuisine_description,
    CAST(inspection_date AS TIMESTAMP)         AS inspection_date,
    violation_code,
    violation_description,
    critical_flag,
    score,
    grade
  FROM workspace.default.restaurant_inspections
  WHERE INITCAP(TRIM(boro)) IN (
    'Manhattan',
    'Brooklyn',
    'Queens',
    'Bronx',
    'Staten Island'
  )
    AND zipcode IS NOT NULL
    AND zipcode BETWEEN 10001 AND 11697
    AND CAST(inspection_date AS TIMESTAMP)
          >= TIMESTAMP('1901-01-01 00:00:00')
)

SELECT
  *,
  CONCAT(
    restaurant_id,
    '_',
    DATE_FORMAT(inspection_date, 'yyyy-MM-dd')
  ) AS inspection_id,

  violation_code = '04K'
    AS rat_violation,

  violation_code = '04L'
    AS mouse_violation,

  violation_code IN ('04K', '04L')
    AS rodent_violation

FROM filtered_and_deduplicated;

WITH filtered_raw AS (
  SELECT
    CAST(camis AS STRING)                      AS restaurant_id,
    dba                                        AS restaurant_name,
    INITCAP(TRIM(boro))                        AS borough,
    LPAD(CAST(zipcode AS STRING), 5, '0')      AS zip_code,
    cuisine_description,
    CAST(inspection_date AS TIMESTAMP)         AS inspection_date,
    violation_code,
    violation_description,
    critical_flag,
    score,
    grade
  FROM workspace.default.restaurant_inspections
  WHERE INITCAP(TRIM(boro)) IN (
    'Manhattan',
    'Brooklyn',
    'Queens',
    'Bronx',
    'Staten Island'
  )
    AND zipcode IS NOT NULL
    AND zipcode BETWEEN 10001 AND 11697
    AND CAST(inspection_date AS TIMESTAMP)
          >= TIMESTAMP('1901-01-01 00:00:00')
)

SELECT
  (SELECT COUNT(*) FROM filtered_raw)
    AS filtered_source_rows,

  (SELECT COUNT(*) FROM workspace.default.inspections_clean_v2)
    AS deduplicated_rows,

  (SELECT COUNT(*) FROM filtered_raw)
    -
  (SELECT COUNT(*) FROM workspace.default.inspections_clean_v2)
    AS exact_duplicate_rows_removed;

SELECT
  COUNT(*) AS violation_rows,
  COUNT(DISTINCT restaurant_id) AS distinct_restaurants,
  COUNT(DISTINCT inspection_id) AS distinct_inspection_events
FROM workspace.default.inspections_clean_v2;

SELECT
  violation_code,
  MAX(violation_description) AS example_description,
  COUNT(*) AS violation_rows,
  COUNT(DISTINCT restaurant_id) AS distinct_restaurants,
  COUNT(DISTINCT inspection_id) AS distinct_inspection_events
FROM workspace.default.inspections_clean_v2
WHERE violation_code IN ('04K', '04L')
GROUP BY violation_code
ORDER BY violation_code;
